# Inspect the current BM25 inverted index

This notebook inspects the active SQLite FTS5 BM25 index. The source database is opened read-only. A temporary copy is used only to create the `fts5vocab` inspection table, so the production index is not modified.

In [4]:
from pathlib import Path
import json, shutil, sqlite3, tempfile

ROOT = Path.cwd()
if not (ROOT / 'apps').exists(): ROOT = Path('/Users/nak/CADT/projects/legal-rag')
INDEX = ROOT / 'data/dense_retrieval/bm25.sqlite3'
if not INDEX.exists(): raise FileNotFoundError(INDEX)

read_only = sqlite3.connect(f'file:{INDEX}?mode=ro', uri=True)
metadata = dict(read_only.execute('SELECT key, value FROM index_metadata'))
print('Index:', INDEX)
print('Size MB:', round(INDEX.stat().st_size / 1024 / 1024, 3))
print('Metadata:')
for key, value in sorted(metadata.items()): print(f'  {key}: {value}')
print('Documents:', read_only.execute('SELECT COUNT(*) FROM documents').fetchone()[0])
print('Filtered terms:', read_only.execute('SELECT COUNT(*) FROM filtered_terms').fetchone()[0])
print('Filtered term examples:', [r[0] for r in read_only.execute('SELECT term FROM filtered_terms ORDER BY term LIMIT 30')])
read_only.close()

Index: /Users/nak/CADT/projects/legal-rag/data/dense_retrieval/bm25.sqlite3
Size MB: 6.273
Metadata:
  articles_sha256: 48e0749c85aec163b377f68065b0a16e0cc56110b21e3d1e450e5cc8c96f457c
  b: 0.75
  bm25_version: sqlite-fts5-v1
  chunks_sha256: 2ac4b62c3222727e26f21718b23b6776a719d49edc497186d9424db423c07e2c
  document_count: 3505
  high_df_ratio: 0.95
  high_df_term_count: 0
  k1: 1.5
  tokenizer_version: khmer-word-v6-kswv2-core-ascii-highdf95
Documents: 3505
Filtered terms: 0
Filtered term examples: []


In [5]:
# Create an inspection-only copy so fts5vocab can be created safely.
with tempfile.TemporaryDirectory(prefix='legalrag-bm25-inspect-') as temp_dir:
    temp_index = Path(temp_dir) / 'bm25.sqlite3'
    shutil.copy2(INDEX, temp_index)
    connection = sqlite3.connect(temp_index)
    connection.execute("CREATE VIRTUAL TABLE vocab USING fts5vocab(documents_fts, 'row')")
    document_count = connection.execute('SELECT COUNT(*) FROM documents').fetchone()[0]
    vocabulary_count = connection.execute('SELECT COUNT(*) FROM vocab').fetchone()[0]
    posting_count = connection.execute('SELECT COALESCE(SUM(doc), 0) FROM vocab').fetchone()[0]
    occurrence_count = connection.execute('SELECT COALESCE(SUM(cnt), 0) FROM vocab').fetchone()[0]
    print('Vocabulary terms:', vocabulary_count)
    print('Distinct postings:', posting_count)
    print('Token occurrences:', occurrence_count)
    print('Top terms by document frequency:')
    for term, doc, cnt in connection.execute('SELECT term, doc, cnt FROM vocab ORDER BY doc DESC, cnt DESC LIMIT 50'):
        print(f'{term!r:30} DF={doc:5} ({doc/document_count:.2%}) occurrences={cnt}')
    connection.close()

Vocabulary terms: 6465
Distinct postings: 135942
Token occurrences: 211919
Top terms by document frequency:
'ស្តី'                         DF= 3232 (92.21%) occurrences=3885
'ច្បាប់'                       DF= 2964 (84.56%) occurrences=4695
'បាន'                          DF= 2104 (60.03%) occurrences=5083
'មិន'                          DF= 1489 (42.48%) occurrences=2711
'ឆ្នាំ'                        DF= 1042 (29.73%) occurrences=1747
'សិទ្ធិ'                       DF=  918 (26.19%) occurrences=1826
'ការងារ'                       DF=  912 (26.02%) occurrences=2112
'ពាណិជ្ជកម្ម'                  DF=  905 (25.82%) occurrences=1822
'កំណត់'                        DF=  896 (25.56%) occurrences=1416
'ដូច'                          DF=  887 (25.31%) occurrences=1247
'១'                            DF=  776 (22.14%) occurrences=1194
'ចែង'                          DF=  775 (22.11%) occurrences=1124
'អនុវត្ត'                      DF=  767 (21.88%) occurrences=1206
'២'                            DF=

In [6]:
# Inspect the physical FTS5 tables and sample documents matching selected terms.
connection = sqlite3.connect(f'file:{INDEX}?mode=ro', uri=True)
tables = connection.execute("SELECT name FROM sqlite_master WHERE name LIKE 'documents_fts%' ORDER BY name").fetchall()
print('FTS5 internal tables:')
for (name,) in tables:
    if name == 'documents_fts':
        print('  documents_fts: virtual table')
        continue
    try:
        count = connection.execute(f'SELECT COUNT(*) FROM [{name}]').fetchone()[0]
        print(f'  {name}: {count:,} rows')
    except sqlite3.OperationalError as error:
        print(f'  {name}: not directly countable ({error})')

def sample_matches(term, limit=10):
    escaped = term.replace(chr(34), '')
    return connection.execute(
        "SELECT documents.chunk_id, documents.article_id, substr(documents.text, 1, 160)"
        " FROM documents_fts JOIN documents ON documents.row_id = documents_fts.rowid"
        " WHERE documents_fts MATCH ? LIMIT ?",
        (f'\"{escaped}\"', limit),
    ).fetchall()

for term in ['ស', 'ច្បាប់', 'មាត្រា', '២០១៧']:
    print(f'\nMatches for {term!r}:')
    try:
        for row in sample_matches(term): print(row)
    except sqlite3.OperationalError as error:
        print('No valid FTS match:', error)
connection.close()

FTS5 internal tables:
  documents_fts: virtual table
  documents_fts_config: 1 rows
  documents_fts_data: 143 rows
  documents_fts_docsize: 3,505 rows
  documents_fts_idx: 135 rows

Matches for 'ស':
('1-article-1-chunk-1', '1-article-1', 'ចរាចរណ៍ផ្លូវគោក\nដែលត្រូវប្រកាសឱ្យប្រើដោយព្រះរាជក្រម\nលេខ នស/រកម/០១១៧/០០២\nចុះថ្ងៃទី ២៦ ខែ មករា ឆ្នាំ ២០១៧')
('1006-article-7-chunk-1', '1006-article-7', 'ស្នាដៃទាំងឡាយ យោងតាមមាត្រា ៣ ដែលត្រូវទទួលបានការការពារក្រោមច្បាប់នេះមាន ៖\nក-សៀវភៅអានគ្រប់ប្រភេទ ឬឯកសារផ្សេងៗទៀតស្តីពីអក្សរសាស្ត្រ សិល្បៈ វិទ្យាសាស្ត្រ និងការអប់រ')
('1013-article-61-chunk-1', '1013-article-61', 'នៅក្នុងច្បាប់នេះ :\n-ចំណាត់ថ្នាក់អន្តរជាតិ មានន័យថា ចំណាត់ថ្នាក់ដែលអនុលោមតាមកិច្ចព្រមព្រៀងទីក្រុងនីស ដែលទាក់ទងអំពីចំណាត់ថ្នាក់អន្តរជាតិនៃទំនិញនិងសេវាកម្ម សម្រាប់')
('1055-article-3-chunk-1', '1055-article-3', 'របបសន្តិសុខសង្គម ដែលបង្កើតដោយច្បាប់នេះត្រូវធានារ៉ាប់រងដោយបេឡាជាតិរបបសន្តិសុខសង្គម សរសេរកាត់ថា “ប.ស.ស” ៖ បេឡាជាតិរបបសន្តិសុខសង្គមជាគ្រឹះស្ថានសាធារណៈ ដែលត្រូវបង្ក')
('1055-article-6-chun

## Interpretation

- `vocab.doc` is document frequency.
- `vocab.cnt` is total token occurrence count.
- `documents_fts_data` and `documents_fts_idx` contain the physical FTS5 index structures.
- `filtered_terms` contains terms removed by the corpus DF filter.
- If frequent entries appear as single Khmer characters, the FTS5 `unicode61` tokenizer is splitting Khmer at character-like units; deleting those entries would be risky.